In [1]:
import requests
import pandas as pd

# Test the API with a simple request - 2023 race results
url = "https://api.jolpi.ca/ergast/f1/2023/results.json"
response = requests.get(url)

print("Status Code:", response.status_code)
data = response.json()
print(data.keys())

Status Code: 200
dict_keys(['MRData'])


In [3]:
import time

all_results = []

for year in range(2019, 2024):
    for round_num in range(1, 24):  # up to 23 races per season
        url = f"https://api.jolpi.ca/ergast/f1/{year}/{round_num}/results.json"
        response = requests.get(url)
        
        if response.status_code != 200:
            continue
        
        data = response.json()
        races = data['MRData']['RaceTable']['Races']
        
        if not races:
            continue
        
        race = races[0]
        for result in race['Results']:
            all_results.append({
                'season': year,
                'round': round_num,
                'race_name': race['raceName'],
                'circuit': race['Circuit']['circuitName'],
                'date': race['date'],
                'driver': result['Driver']['givenName'] + ' ' + result['Driver']['familyName'],
                'constructor': result['Constructor']['name'],
                'grid': int(result['grid']),
                'position': result.get('position'),
                'points': float(result['points']),
                'status': result['status'],
                'laps': int(result['laps'])
            })
        
        time.sleep(0.3)  # be polite to the free API, avoid rate limiting
    
    print(f"Finished season {year}")

print(f"\nTotal results collected: {len(all_results)}")

Finished season 2019
Finished season 2020
Finished season 2021
Finished season 2022
Finished season 2023

Total results collected: 2080


In [4]:
results_df = pd.DataFrame(all_results)
print(results_df.shape)
results_df.head(10)

(2080, 12)


,season,round,race_name,circuit,date,driver,constructor,grid,position,points,status,laps
0,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Valtteri Bottas,Mercedes,2,1,26.0,Finished,58
1,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Lewis Hamilton,Mercedes,1,2,18.0,Finished,58
2,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Max Verstappen,Red Bull,4,3,15.0,Finished,58
3,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Sebastian Vettel,Ferrari,3,4,12.0,Finished,58
4,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Charles Leclerc,Ferrari,5,5,10.0,Finished,58
5,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Kevin Magnussen,Haas F1 Team,7,6,8.0,Finished,58
6,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Nico Hülkenberg,Renault,11,7,6.0,+1 Lap,57
7,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Kimi Räikkönen,Alfa Romeo,9,8,4.0,+1 Lap,57
8,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Lance Stroll,Racing Point,16,9,2.0,+1 Lap,57
9,2019,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2019-03-17,Daniil Kvyat,Toro Rosso,15,10,1.0,+1 Lap,57


In [5]:
results_df.to_csv('../data/race_results.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [6]:
winners = results_df[results_df['position'] == '1']
top_winners = winners['driver'].value_counts().head(10)
top_winners

driver
Max Verstappen      49
Lewis Hamilton      30
Valtteri Bottas      7
Sergio Pérez         6
Charles Leclerc      5
Carlos Sainz         2
Sebastian Vettel     1
Pierre Gasly         1
Esteban Ocon         1
Daniel Ricciardo     1
Name: count, dtype: int64